In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.3 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO
from collections import deque, defaultdict
import os
import glob

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
YOLO_PATH = '/content/drive/MyDrive/CapstoneProject/models/model_v8n_ver6/yolo/weights/best.pt'
RL_MODEL_PATH = '/content/drive/MyDrive/CapstoneProject/RL/rl_agent_final.pth'

In [ ]:
VIDEO_PATH = "/content/drive/MyDrive/CapstoneProject/make/data/batch_42"
OUTPUT_PATH = '/content/drive/MyDrive/CapstoneProject/test/result_rl_final.mp4'

In [ ]:
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.fc4(x)

class InferenceAgent:
    def __init__(self, model_path, state_size=50, action_size=2):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = DQN(state_size, action_size).to(self.device)

        if os.path.exists(model_path):
            self.model.load_state_dict(torch.load(model_path, map_location=self.device))
            self.model.eval()
            print(f"Đã load RL Agent: {model_path}")
        else:
            raise FileNotFoundError(f"Không tìm thấy file model RL tại: {model_path}")

    def act(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.model(state_t)
        return torch.argmax(q_values).item()

def create_state(history_queue):
    state = np.array(history_queue).flatten()
    if len(state) < 50:
        padding = np.zeros(50 - len(state))
        state = np.concatenate((padding, state))
    return state

In [ ]:
yolo = YOLO(YOLO_PATH)
agent = InferenceAgent(RL_MODEL_PATH)

source_type = 'video'
image_files = []
current_idx = 0

if os.path.isdir(VIDEO_PATH):
    source_type = 'folder'
    image_files = sorted(glob.glob(os.path.join(VIDEO_PATH, '*.jpg')))
    if not image_files: raise ValueError("Folder rỗng!")
    first_frame = cv2.imread(image_files[0])
    height, width = first_frame.shape[:2]
    fps = 30
else:
    cap = cv2.VideoCapture(VIDEO_PATH)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

Đã load RL Agent: /content/drive/MyDrive/CapstoneProject/RL/rl_agent_final.pth


In [ ]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

In [ ]:
track_history = defaultdict(lambda: deque(maxlen=10))
STATE_WINDOW = 10

while True:
    if source_type == 'video':
        ret, frame = cap.read()
        if not ret: break
    else:
        if current_idx >= len(image_files): break
        frame = cv2.imread(image_files[current_idx])
        current_idx += 1
        if frame is None: continue

    img_h, img_w = frame.shape[:2]

    results = yolo.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        clss = results[0].boxes.cls.cpu().numpy().astype(int)

        for box, track_id, conf, cls in zip(boxes, ids, confs, clss):
            if cls == 0 or cls == 1:
                continue

            x1, y1, x2, y2 = box

            cx = ((x1 + x2) / 2) / img_w
            cy = ((y1 + y2) / 2) / img_h
            w_norm = (x2 - x1) / img_w
            h_norm = (y2 - y1) / img_h

            feature = [cx, cy, w_norm, h_norm, float(conf)]

            track_history[track_id].append(feature)

            color = (0, 255, 0)
            label = "Safe"

            if len(track_history[track_id]) == STATE_WINDOW:
                state_vector = create_state(track_history[track_id])
                action = agent.act(state_vector)

                if action == 1:
                    color = (0, 0, 255)
                    label = "CUT-IN!"

                    cv2.putText(frame, "WARNING: CUT-IN!", (50, 100),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            text_size = cv2.getTextSize(f"ID:{track_id} {label}", cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
            cv2.rectangle(frame, (x1, y1 - 20), (x1 + text_size[0], y1), color, -1)

            cv2.putText(frame, f"ID:{track_id} {label}", (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    out.write(frame)

if source_type == 'video':
    cap.release()
out.release()
print(f"Xử lý hoàn tất! Video lưu tại: {OUTPUT_PATH}")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 123ms
Prepared 1 package in 20ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Xử lý hoàn tất! Video lưu tại: /content/drive/MyDrive/CapstoneProject/test/result_rl_final.mp4
